# EDA Report — Predição de Abandono do Tratamento de TB
**Macro 2** | Amostra: `amostra_2025.xlsx` (500 registros, 94 colunas) | SINAN-TB 2025

**Entregável:** Atividades 2.3, 2.4, 2.5 e 2.6 da Macro 2  
**Responsáveis:** Gustavo (pipeline/EDA) + Membro 3 (documentação)  
**Prazo:** 11/05

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#444',
    'axes.labelcolor': '#ccc',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'text.color': '#eee',
    'grid.color': '#333',
    'grid.alpha': 0.5,
    'axes.grid': True
})

# Ajustar path conforme necessário
df = pd.read_excel('../amostra_2025.xlsx')
print(f'Shape: {df.shape}')
print(f'Registros: {df.shape[0]} | Variáveis: {df.shape[1]}')

## 1. Visão Geral — Qualidade dos Dados (Atividade 2.3)

In [ ]:
miss = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
drop100 = miss[miss == 100]
drop80 = miss[(miss > 80) & (miss < 100)]
drop50 = miss[(miss > 50) & (miss <= 80)]

print(f'Colunas 100% nulas (drop imediato): {len(drop100)}')
print(list(drop100.index))
print(f'\nColunas 80-100% nulas (avaliar): {len(drop80)}')
print(list(drop80.index))
print(f'\nColunas 50-80% nulas: {len(drop50)}')
print(list(drop50.index))
print(f'\nColunas sem nulos: {(miss == 0).sum()}')

In [ ]:
miss_plot = miss[miss > 0]
colors = ['#e74c3c' if v == 100 else '#e67e22' if v > 80 else '#f1c40f' if v > 50 else '#3498db' for v in miss_plot]

fig, ax = plt.subplots(figsize=(14, 9))
ax.barh(range(len(miss_plot)), miss_plot.values, color=colors)
ax.set_yticks(range(len(miss_plot)))
ax.set_yticklabels(miss_plot.index, fontsize=8)
ax.axvline(50, color='#e67e22', ls='--', alpha=0.7, label='50%')
ax.axvline(80, color='#e74c3c', ls='--', alpha=0.7, label='80%')
ax.set_xlabel('% Nulos')
ax.set_title('Missingness por Variável (apenas colunas com nulos)')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Variável-Alvo — SITUA_ENCE (Atividade 2.4)

In [ ]:
labels_enc = {
    1.0: 'Cura',
    2.0: 'Abandono',
    3.0: 'Transferência',
    4.0: 'Óbito outras causas',
    5.0: 'Óbito por TB',
    7.0: 'TB-DR',
    8.0: 'Mudança esquema',
    10.0: 'Falência'
}
vc = df['SITUA_ENCE'].value_counts(dropna=False).sort_index()
print('Distribuição SITUA_ENCE:')
for k, v in vc.items():
    lbl = labels_enc.get(k, 'Sem encerramento') if not pd.isna(k) else 'Sem encerramento'
    print(f'  {str(k):5s} {lbl:25s}: {v:4d} ({v/len(df)*100:.1f}%)')

# Criar variável-alvo
df['ltfu'] = df['SITUA_ENCE'].map({1.0: 0, 2.0: 1})
labeled = df['ltfu'].dropna()
print(f'\nRegistros rotulados (cura+abandono): {len(labeled)} ({len(labeled)/len(df)*100:.1f}%)')
print(f'LTFU=1 (abandono): {int(labeled.sum())} ({labeled.mean()*100:.1f}%)')
print(f'LTFU=0 (cura):     {int((labeled==0).sum())} ({(labeled==0).mean()*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza todos os valores
vc_plot = df['SITUA_ENCE'].value_counts(dropna=False)
pie_labels = [labels_enc.get(k, 'Sem enc.') if not pd.isna(k) else 'Sem enc.' for k in vc_plot.index]
pie_colors = ['#1D9E75', '#e74c3c', '#7F77DD', '#e67e22', '#BA7517', '#f1c40f', '#aaa', '#5dade2', '#888780']
axes[0].pie(vc_plot.values, labels=pie_labels, autopct='%1.0f%%',
            colors=pie_colors[:len(vc_plot)], textprops={'color': '#eee'}, startangle=90)
axes[0].set_title('SITUA_ENCE — Todos os Valores')

# Barras classes rotuladas
axes[1].bar(['Cura (ltfu=0)', 'Abandono (ltfu=1)'],
            [int((labeled==0).sum()), int(labeled.sum())],
            color=['#1D9E75', '#e74c3c'])
axes[1].set_title('Classes Rotuladas — Balanceamento')
axes[1].set_ylabel('Contagem')

plt.suptitle('Variável-Alvo: SITUA_ENCE → ltfu', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Análise Univariada — Variáveis Demográficas e Clínicas (Atividade 2.4)

In [ ]:
# Decodificar idade SINAN: prefixo 4=anos, 3=meses, 2=dias
df['nu_str'] = df['NU_IDADE_N'].astype(str).str.zfill(4)
df['idade_unid'] = pd.to_numeric(df['nu_str'].str[0], errors='coerce')
df['idade_val'] = pd.to_numeric(df['nu_str'].str[1:], errors='coerce')
df['idade_anos'] = np.where(
    df['idade_unid'] == 4, df['idade_val'],
    np.where(df['idade_unid'] == 3, (df['idade_val'] / 12).round(),
    np.where(df['idade_unid'] == 2, (df['idade_val'] / 365).round(), np.nan))
)
print('Estatísticas de Idade:')
print(df['idade_anos'].describe().round(1))

In [ ]:
raca_map = {1: 'Branca', 2: 'Preta', 3: 'Amarela', 4: 'Parda', 5: 'Indígena', 9: 'Ignorado'}
esc_map = {0: 'Analfabeto', 1: '1ª-4ª', 2: '5ª-8ª', 3: 'Médio', 4: 'Superior', 5: 'N/A', 9: 'Ignorado'}
forma_map = {1: 'Pulmonar', 2: 'Extrapulmonar', 3: 'Pulm+Extrapulm'}
trat_map = {1: 'Caso novo', 2: 'Recidiva', 3: 'Reingresso\npós-abandon.', 4: 'Não sabe', 5: 'Transferência', 6: 'Pós-óbito'}

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Idade
axes[0, 0].hist(df['idade_anos'].dropna(), bins=20, color='#3498db', edgecolor='#0f0f0f')
axes[0, 0].set_title('Distribuição de Idade')
axes[0, 0].set_xlabel('Idade (anos)')

# Sexo
vc_sex = df['CS_SEXO'].value_counts()
axes[0, 1].bar(vc_sex.index, vc_sex.values, color=['#3498db', '#e91e8c'])
axes[0, 1].set_title('Sexo')
for i, (k, v) in enumerate(vc_sex.items()):
    axes[0, 1].text(i, v + 2, str(v), ha='center', color='#eee')

# Raça
vc_raca = df['CS_RACA'].map(raca_map).value_counts()
axes[0, 2].barh(vc_raca.index, vc_raca.values, color='#9b59b6')
axes[0, 2].set_title('Raça/Cor')

# Escolaridade
vc_esc = df['CS_ESCOL_N'].map(esc_map).value_counts()
axes[1, 0].barh(vc_esc.index, vc_esc.values, color='#1abc9c')
axes[1, 0].set_title('Escolaridade')

# Forma clínica
vc_forma = df['FORMA'].map(forma_map).value_counts()
axes[1, 1].bar(vc_forma.index, vc_forma.values, color=['#e67e22', '#e74c3c', '#f39c12'])
axes[1, 1].set_title('Forma Clínica')

# Tipo de entrada
vc_trat = df['TRATAMENTO'].map(trat_map).value_counts()
axes[1, 2].barh(vc_trat.index, vc_trat.values, color='#e74c3c')
axes[1, 2].set_title('Tipo de Entrada (TRATAMENTO)')

plt.suptitle('Variáveis Demográficas e Clínicas — Distribuições', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Análise Bivariada — Taxa de Abandono por Grupo (Atividade 2.4)

In [ ]:
df_labeled = df[df['ltfu'].notna()].copy()

vars_biv = [
    ('CS_SEXO',    None,                                       'Sexo'),
    ('HIV',        {1: 'Positivo', 2: 'Negativo', 3: 'Andamento', 4: 'Não realiz.'}, 'HIV'),
    ('AGRAVAIDS',  {1: 'Sim', 2: 'Não', 9: 'Ignorado'},       'Agravo AIDS'),
    ('AGRAVALCOO', {1: 'Sim', 2: 'Não', 9: 'Ignorado'},       'Alcoolismo'),
    ('AGRAVDROGA', {1: 'Sim', 2: 'Não', 9: 'Ignorado'},       'Uso de Drogas'),
    ('POP_RUA',    {1: 'Sim', 2: 'Não', 9: 'Ignorado'},       'Pop. em Situação de Rua'),
    ('POP_LIBER',  {1: 'Sim', 2: 'Não', 9: 'Ignorado'},       'Pop. Privada de Liberdade'),
    ('TRATAMENTO', {1: 'Caso novo', 2: 'Recidiva', 3: 'Reingresso', 5: 'Transf.'}, 'Tipo de Entrada'),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
axes = axes.flatten()

for i, (col, mapa, title) in enumerate(vars_biv):
    col_data = df_labeled[col].copy()
    if mapa:
        col_data = col_data.map(mapa)
    taxa = df_labeled.groupby(col_data)['ltfu'].mean() * 100
    n_group = df_labeled.groupby(col_data)['ltfu'].count()
    bars = axes[i].bar(taxa.index.astype(str), taxa.values, color='#e74c3c', alpha=0.85)
    axes[i].set_title(title, fontsize=10)
    axes[i].set_ylabel('% Abandono')
    axes[i].set_ylim(0, 100)
    for bar, cnt in zip(bars, n_group.values):
        axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                     f'n={cnt}', ha='center', fontsize=8, color='#aaa')
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('Taxa de Abandono (%) por Grupo — Análise Bivariada', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Prevalência de Comorbidades e Vulnerabilidades

In [ ]:
comorbidades = ['AGRAVAIDS', 'AGRAVALCOO', 'AGRAVDIABE', 'AGRAVDOENC', 'AGRAVDROGA', 'AGRAVTABAC', 'POP_RUA', 'POP_LIBER', 'POP_IMIG', 'BENEF_GOV']
labels_c =     ['AIDS',      'Alcoolismo', 'Diabetes',   'D. Mental',  'Drogas',     'Tabagismo',  'Sit. Rua',  'Priv. Lib.', 'Imigrante', 'Benef. Gov.']
prev = [(df[c] == 1).sum() / df[c].notna().sum() * 100 for c in comorbidades]

fig, ax = plt.subplots(figsize=(13, 5))
bar_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#9b59b6', '#c0392b', '#95a5a6', '#1abc9c', '#3498db', '#27ae60', '#d35400']
bars = ax.bar(labels_c, prev, color=bar_colors, alpha=0.87)
for bar, v in zip(bars, prev):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f'{v:.1f}%', ha='center', fontsize=9, color='#eee')
ax.set_ylabel('Prevalência (%)')
ax.set_title('Prevalência de Comorbidades e Vulnerabilidades na Amostra')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 6. Decisões de Qualidade e Drop — Atividade 2.5

In [ ]:
decisions = [
    ('RIFAMPICIN, ISONIAZIDA, ETAMBUTOL, ESTREPTOMI, PIRAZINAMI, ETIONAMIDA, OUTRAS', '100%', 'DROP', 'Campos de medicamentos não preenchidos neste dataset'),
    ('BACILOS_E2, DT_MUDANCA, SITUA_9_M, SITUA_12_M, DOENCA_TRA', '100%', 'DROP', '100% nulos — sem informação útil'),
    ('DT_TRANSRM, CS_FLXRET', '100%', 'DROP', 'Campos administrativos completamente vazios'),
    ('DT_ENCERRA', '41.8%', 'DROP', 'Data leakage: ocorre após o desfecho ser definido'),
    ('BACILOSC_1 ... BACILOSC_6', '40–65%', 'DROP (modelo base)', 'Data leakage: exames de acompanhamento durante tratamento'),
    ('BAC_APOS_6', '81.2%', 'DROP', 'Pós-tratamento — leakage direto'),
    ('UF_TRANSF, MUN_TRANSF', '>93%', 'DROP', 'Quase completamente vazios'),
    ('TP_NOT, ID_AGRAVO, NU_ANO, DT_DIGITA, NDUPLIC_N', '0%', 'DROP (admin)', 'Identificadores sem valor preditivo'),
    ('TRAT_SUPER', '79%', 'AVALIAR', 'Preditor importante — missingness alta na amostra 2025'),
    ('ANT_RETRO', '70.8%', 'AVALIAR', 'Alta missingness; relevante apenas para HIV+'),
    ('CULTURA_OU, BACILOSC_O', '>79%', 'AVALIAR', 'Alta missingness; versão "escarro" disponível'),
    ('TRANSF', '91%', 'AVALIAR', 'Quase vazio; redundante com SITUA_ENCE=3'),
]

df_dec = pd.DataFrame(decisions, columns=['Coluna(s)', '% Nulo', 'Decisão', 'Justificativa'])
pd.set_option('display.max_colwidth', 60)
print(df_dec.to_string(index=False))

## 7. Resumo de Insights — Guia para Macro 3 (Atividade 2.6)

### Achados Principais

| # | Insight | Impacto para Macro 3 |
|---|---------|---------------------|
| 1 | **Classes balanceadas** nos rotulados: 97 curas vs 97 abandonos (50/50) | SMOTE pode não ser necessário |
| 2 | **37% sem encerramento** (NaN em SITUA_ENCE) — casos 2025 em andamento | Excluir conforme data-prep.py |
| 3 | **TRATAMENTO=3** (reingresso pós-abandono): 41% da amostra | Feature binária `reingresso` |
| 4 | **Sexo**: 70% masculino — homens com maior risco histórico | Incluir como feature |
| 5 | **Forma clínica**: 83% pulmonar — filtro correto | Filtrar antes do pipeline |
| 6 | **NU_IDADE_N** requer decodificação (prefixo 4=anos, 3=meses, 2=dias) | Gerar `idade_anos` |
| 7 | **Valor 9** em binárias = ignorado/NI no SINAN | Tratar como categoria ou imputar moda |
| 8 | **10 colunas 100% nulas** → drop imediato | Reduz de 94 para ~84 colunas |
| 9 | **TRAT_SUPER** 79% nulo na amostra 2025 | Verificar no dataset completo |
| 10 | **DT_ENCERRA + BACILOSC_1–6** revelam o desfecho | Drop obrigatório (leakage) |

### Próximos Passos — Macro 3 (Feature Engineering)

- Decodificar `NU_IDADE_N` → `idade_anos`
- Feature `reingresso` = (`TRATAMENTO` == 3).astype(int)
- Feature `dias_inicio_trat` = `DT_INIC_TR` - `DT_NOTIFIC`
- Tratar valor 9 como categoria em `AGRAVAIDS`, `AGRAVALCOO`, `AGRAVDROGA`, `POP_RUA`, `POP_LIBER`
- Encoding ordinal para `CS_ESCOL_N`
- One-hot encoding para `CS_RACA`, `HIV`, `FORMA`, `BACILOSC_E`, `CULTURA_ES`
- Avaliar `TRAT_SUPER` e `BENEF_GOV` no dataset completo (missingness pode ser menor)
